# Coding Exercise: Structural Topology Optimization from Scratch

Topology optimization is a mathematical approach that optimizes material layout within a given design space, for a given set of loads, boundary conditions, and constraints. The goal is to maximize the performance (e.g., stiffness) of the system.

In this exercise, you will implement a 2D finite element method (FEM) solver in Python based on the Solid Isotropic Material with Penalization (SIMP) model, and use it inside a gradient-based optimization loop to design an optimal structure.

---

## 1. Problem Formulation & Physics

Consider a 2D design domain discretized into a grid of $N_x \times N_y$ rectangular finite elements. The density of each element is represented by a design variable $x_i \in [x_{\min}, 1]$.

Under the **SIMP** model, the Young's modulus $E_i$ of element $i$ is interpolated as:
$$E_i(x_i) = E_{\min} + x_i^p (E_0 - E_{\min})$$
where:
- $E_0$ is the stiffness of the solid material.
- $E_{\min}$ is a tiny stiffness ($10^{-9}$) to avoid a singular global stiffness matrix.
- $p$ is the penalization power (typically $p = 3$).

The global stiffness matrix $\mathbf{K}(\mathbf{x})$ is assembled from the element stiffness matrices:
$$\mathbf{K}(\mathbf{x}) = \sum_{i=1}^N E_i(x_i) \mathbf{K}_0$$
where $\mathbf{K}_0$ is the stiffness matrix of a solid 4-node quad element.

The physical state of the structure is determined by the static equilibrium equation:
$$\mathbf{K}(\mathbf{x}) \mathbf{u} = \mathbf{f}$$
where $\mathbf{u}$ is the displacement vector and $\mathbf{f}$ is the external force vector.

The structural **compliance** (which is twice the total strain energy, and represents the deformability) is defined as:
$$C(\mathbf{x}) = \mathbf{f}^T \mathbf{u}(\mathbf{x}) = \mathbf{u}^T \mathbf{K}(\mathbf{x}) \mathbf{u}$$

### Optimization Formulation:
$$\begin{aligned}
\min_{\mathbf{x}} \quad & C(\mathbf{x}) = \mathbf{f}^T \mathbf{u}(\mathbf{x}) \\
\text{subject to} \quad & \mathbf{K}(\mathbf{x}) \mathbf{u} = \mathbf{f} \\
& \frac{\sum_{i=1}^N x_i v_i}{V_0} \le f \\
& x_{\min} \le x_i \le 1, \quad i = 1, \dots, N
\end{aligned}$$

---

## 2. Theoretical Questions (to think about)

1. **Convexity of $p=1$:** If $p=1$ and $E_{\min}=0$, show that compliance $C(\mathbf{x})$ is a convex function of $\mathbf{x}$ for $\mathbf{x} > \mathbf{0}$. (Hint: the matrix inverse function $A \mapsto c^T A^{-1} c$ is convex on the set of positive-definite matrices).
2. **Non-Convexity of $p=3$:** Why do we use $p=3$ if it makes the optimization problem non-convex and introduces local minima? (Hint: consider the densities of the final solution).
3. **Adjoint Gradients:** Prove that the gradient of compliance w.r.t element density is:
$$\frac{\partial C}{\partial x_i} = - p x_i^{p-1} (E_0 - E_{\min}) \mathbf{u}_i^T \mathbf{K}_0 \mathbf{u}_i$$
Why is this adjoint method much faster than finite differences?


## 3. Programming Implementation


In [ ]:
# A Tutorial on Structural Optimization | Sam Greydanus | 2022
import time, functools, nlopt
import numpy as np                                                # for dense matrix ops
import matplotlib.pyplot as plt                                   # for plotting
import autograd, autograd.core, autograd.extend, autograd.tracer  # for adjoints
import autograd.numpy as anp      
import scipy, scipy.ndimage, scipy.sparse, scipy.sparse.linalg    # sparse matrices


In [ ]:
##### Problem setup helper functions #####
class ObjectView(object):
    def __init__(self, d): self.__dict__ = d
    
def get_args(normals, forces, density=0.4):  # Manage the problem setup parameters
  width = normals.shape[0] - 1
  height = normals.shape[1] - 1
  fixdofs = np.flatnonzero(normals.ravel())
  alldofs = np.arange(2 * (width + 1) * (height + 1))
  freedofs = np.sort(list(set(alldofs) - set(fixdofs)))
  params = {
      # material properties
      'young': 1, 'young_min': 1e-9, 'poisson': 0.3, 'g': 0,
      # constraints
      'density': density, 'xmin': 0.001, 'xmax': 1.0,
      # input parameters
      'nelx': width, 'nely': height, 'mask': 1, 'penal': 3.0, 'filter_width': 1,
      'freedofs': freedofs, 'fixdofs': fixdofs, 'forces': forces.ravel(),
      # optimization parameters
      'opt_steps': 80, 'print_every': 10}
  return ObjectView(params)

def mbb_beam(width=80, height=25, density=0.4, y=1, x=0):  # textbook beam example
  normals = np.zeros((width + 1, height + 1, 2))
  normals[-1, -1, y] = 1
  normals[0, :, x] = 1
  forces = np.zeros((width + 1, height + 1, 2))
  forces[0, 0, y] = -1
  return normals, forces, density

def physical_density(x, args, volume_contraint=False, use_filter=True):
  x = args.mask * x.reshape(args.nely, args.nelx)
  return gaussian_filter(x, args.filter_width) if use_filter else x

def mean_density(x, args, volume_contraint=False, use_filter=True):
  return anp.mean(physical_density(x, args, volume_contraint, use_filter)) / anp.mean(args.mask)

##### Caching wrappers for numpy arrays (gives 2x speedup) #####
class _WrappedArray:
  def __init__(self, value): self.value = value
  def __eq__(self, other): return np.array_equal(self.value, other.value)
  def __hash__(self): return hash(repr(self.value.ravel()))

def ndarray_safe_lru_cache(maxsize=128):
  def decorator(func):
    @functools.lru_cache(maxsize)
    def cached_func(*args, **kwargs):
      args = tuple(a.value if isinstance(a, _WrappedArray) else a for a in args)
      kwargs = {k: v.value if isinstance(v, _WrappedArray) else v for k, v in kwargs.items()}
      return func(*args, **kwargs)
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
      args = tuple(_WrappedArray(a) if isinstance(a, np.ndarray) else a for a in args)
      kwargs = {k: _WrappedArray(v) if isinstance(v, np.ndarray) else v for k, v in kwargs.items()}
      return cached_func(*args, **kwargs)
    return wrapper
  return decorator


In [ ]:
##### Student Section: Complete the TODOs #####

def young_modulus(x, e_0, e_min, p=3):
  """
  TODO: Task 1 - Implement the SIMP Young's modulus interpolation equation:
  E(x) = E_min + (x^p) * (E_0 - E_min)
  """
  # YOUR CODE HERE
  raise NotImplementedError()

def compliance(x_phys, u, ke, *, penal=3, e_min=1e-9, e_0=1):
  """
  TODO: Task 2 - Calculate the compliance C of the design structure.
  1. Construct the grid of nodes (n1, n2, n3, n4) corresponding to each element.
  2. Select the nodal displacements 'u_selected' for each element.
  3. Vectorize the computation of elements compliance 'ce' using numpy's einsum:
     ce_e = u_e^T * ke * u_e
  4. Scale by physical stiffness (SIMP interpolation of young modulus) and sum to return the total compliance.
  """
  nely, nelx = x_phys.shape
  ely, elx = anp.meshgrid(range(nely), range(nelx))

  # Compute element node numbers n1, n2, n3, n4:
  # YOUR CODE HERE (replace the lines below)
  n1 = (nely+1)*(elx+0) + (ely+0)
  n2 = (nely+1)*(elx+1) + (ely+0)
  n3 = (nely+1)*(elx+1) + (ely+1)
  n4 = (nely+1)*(elx+0) + (ely+1)
  
  # YOUR CODE HERE: Construct all_ixs representing DOFs of each node (2*n, 2*n+1)
  all_ixs = anp.array([2*n1, 2*n1+1, 2*n2, 2*n2+1, 2*n3, 2*n3+1, 2*n4, 2*n4+1])
  
  u_selected = u[all_ixs]
  
  # YOUR CODE HERE: Compute einsum contractions
  ke_u = anp.einsum('ij,jkl->ikl', ke, u_selected)
  ce = anp.einsum('ijk,ijk->jk', u_selected, ke_u)
  
  C = young_modulus(x_phys, e_0, e_min, p=penal) * ce.T
  return anp.sum(C)

def get_stiffness_matrix(e, nu):
  """
  TODO: Task 3 - Formulate the element stiffness matrix ke of a 2D square quad element.
  k = [1/2-nu/6, 1/8+nu/8, -1/4-nu/12, -1/8+3*nu/8, -1/4+nu/12, -1/8-nu/8, nu/6, 1/8-3*nu/8]
  """
  # YOUR CODE HERE (replace the lines below)
  k = anp.array([1/2-nu/6, 1/8+nu/8, -1/4-nu/12, -1/8+3*nu/8,
                -1/4+nu/12, -1/8-nu/8, nu/6, 1/8-3*nu/8])
  return e/(1-nu**2)*anp.array([[k[0], k[1], k[2], k[3], k[4], k[5], k[6], k[7]],
                               [k[1], k[0], k[7], k[6], k[5], k[4], k[3], k[2]],
                               [k[2], k[7], k[0], k[5], k[6], k[3], k[4], k[1]],
                               [k[3], k[6], k[5], k[0], k[7], k[2], k[1], k[4]],
                               [k[4], k[5], k[6], k[7], k[0], k[1], k[2], k[3]],
                               [k[5], k[4], k[3], k[2], k[1], k[0], k[7], k[6]],
                               [k[6], k[3], k[4], k[1], k[2], k[7], k[0], k[5]],
                               [k[7], k[2], k[1], k[4], k[3], k[6], k[5], k[0]]])

def _get_dof_indices(freedofs, fixdofs, k_xlist, k_ylist):
  """
  TODO: Task 4 - Implement the indexing setup to filter the sparse stiffness matrix
  for free degrees of freedom (ignoring the fixed boundary DOFs).
  """
  # YOUR CODE HERE
  raise NotImplementedError()

@autograd.primitive
def solve_coo(a_entries, a_indices, b, sym_pos=False):
  """
  TODO: Task 5 - Define a custom solver primitive with Autograd.
  """
  # YOUR CODE HERE
  raise NotImplementedError()

def grad_solve_coo_entries(ans, a_entries, a_indices, b, sym_pos=False):
  # YOUR CODE HERE: complete the VJP for solve_coo
  def jvp(grad_ans):
    lambda_ = solve_coo(a_entries, a_indices if sym_pos else a_indices[::-1], grad_ans, sym_pos)
    i, j = a_indices
    return -lambda_[i] * ans[j]
  return jvp
autograd.extend.defvjp(solve_coo, grad_solve_coo_entries,
                       lambda: print('err: gradient undefined'),
                       lambda: print('err: gradient not implemented'))


In [ ]:
##### Physics helper functions (no student edits needed) #####

def get_k(stiffness, ke):
  nely, nelx = stiffness.shape
  ely, elx = anp.meshgrid(range(nely), range(nelx))
  ely, elx = ely.reshape(-1, 1), elx.reshape(-1, 1)

  n1 = (nely+1)*(elx+0) + (ely+0)
  n2 = (nely+1)*(elx+1) + (ely+0)
  n3 = (nely+1)*(elx+1) + (ely+1)
  n4 = (nely+1)*(elx+0) + (ely+1)
  edof = anp.array([2*n1, 2*n1+1, 2*n2, 2*n2+1, 2*n3, 2*n3+1, 2*n4, 2*n4+1])
  edof = edof.T[0]
  x_list = anp.repeat(edof, 8)
  y_list = anp.tile(edof, 8).flatten()

  kd = stiffness.T.reshape(nelx*nely, 1, 1)
  value_list = (kd * anp.tile(ke, kd.shape)).flatten()
  return value_list, y_list, x_list

def displace(x_phys, ke, forces, freedofs, fixdofs, *, penal=3, e_min=1e-9, e_0=1):
  stiffness = young_modulus(x_phys, e_0, e_min, p=penal)
  k_entries, k_ylist, k_xlist = get_k(stiffness, ke)
  index_map, keep, indices = _get_dof_indices(freedofs, fixdofs, k_ylist, k_xlist)
  
  u_nonzero = solve_coo(k_entries[keep], indices, forces[freedofs], sym_pos=True)
  u_values = anp.concatenate([u_nonzero, anp.zeros(len(fixdofs))])
  return u_values[index_map]

@autograd.extend.primitive
def gaussian_filter(x, width):
  return scipy.ndimage.gaussian_filter(x, width, mode='reflect')

def _gaussian_filter_vjp(ans, x, width):
  del ans, x
  return lambda g: gaussian_filter(g, width)
autograd.extend.defvjp(gaussian_filter, _gaussian_filter_vjp)


In [ ]:
##### Optimization solver and runner #####

def fast_stopt(args, x=None, verbose=True):
  if x is None:
    x = anp.ones((args.nely, args.nelx)) * args.density  # init mass

  reshape = lambda x: x.reshape(args.nely, args.nelx)
  objective_fn = lambda x: objective(reshape(x), args)
  constraint = lambda params: mean_density(reshape(params), args) - args.density

  def wrap_autograd_func(func, losses=None, frames=None):
    def wrapper(x, grad):
      if grad.size > 0:
        value, grad[:] = autograd.value_and_grad(func)(x)
      else:
        value = func(x)
      if losses is not None:
        losses.append(value)
      if frames is not None:
        frames.append(reshape(x).copy())
        if verbose and len(frames) % args.print_every == 0:
          print('step {}, loss {:.2e}, t={:.2f}s'.format(len(frames), value, time.time()-dt))
      return value
    return wrapper

  losses, frames = [], [] ; dt = time.time()
  print('Optimizing a problem with {} nodes'.format(len(args.forces)))
  opt = nlopt.opt(nlopt.LD_MMA, x.size)
  opt.set_lower_bounds(0.0) ; opt.set_upper_bounds(1.0)
  opt.set_min_objective(wrap_autograd_func(objective_fn, losses, frames))
  opt.add_inequality_constraint(wrap_autograd_func(constraint), 1e-8)
  opt.set_maxeval(args.opt_steps + 1)
  opt.optimize(x.flatten())
  return np.array(losses), reshape(frames[-1]), np.array(frames)

# Run optimization on standard MBB beam:
args = get_args(*mbb_beam())
losses, x_opt, mbb_frames = fast_stopt(args=args, verbose=True)

# Plot final result:
plt.figure(figsize=(10, 4), dpi=100)
plt.imshow(np.concatenate([x_opt[:,::-1], x_opt], axis=1), cmap='gray_r')
plt.title("Final Optimized MBB Beam")
plt.axis('off')
plt.show()
